# 17_generate_mel_nv_cue_qualitative_pdf

Generate qualitative GradCAM / Map Diff / FinerCAM comparison panels for the controlled binary MEL vs NV synthetic cue experiment.

This notebook uses the cued fixed center MEL/NV test set and compares:

1. Clean MEL/NV CE model
2. Cued MEL/NV CE model
3. Cued MEL/NV + HA model

The target/reference pair is class specific:

- GT = MEL: target MEL, reference NV
- GT = NV: target NV, reference MEL


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

QUAL_SEED = 42
BLOCKS_TO_COMPARE = [-1, -4]   # -1 = last block, -4 = middle block found useful in cue experiment

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"

# 7-class qualitative CSV
QUAL_CSV = HAM_ROOT / f"ham_test_cam_qualitative_stratified_10_seed{QUAL_SEED}.csv"

# Checkpoints for normal 7-class models
CHECKPOINT_ROOT = REPO_ROOT / "external" / "checkpoints3"
BASE_CKPT = CHECKPOINT_ROOT / "checkpoint-best-base.pth"
HA_CKPT = CHECKPOINT_ROOT / "checkpoint-best-HA075.pth"

QUAL_ROOT = REPO_ROOT / "outputs" / f"overview_7class_base_vs_ha_seed{QUAL_SEED}"
QUAL_ROOT.mkdir(parents=True, exist_ok=True)

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

print("QUAL_CSV:", QUAL_CSV)
print("BASE_CKPT:", BASE_CKPT)
print("HA_CKPT:", HA_CKPT)
print("QUAL_ROOT:", QUAL_ROOT)

for p in [QUAL_CSV, BASE_CKPT, HA_CKPT]:
    if not p.exists():
        print("[WARN] missing:", p)

QUAL_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed42.csv
BASE_CKPT: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-base.pth
HA_CKPT: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-HA075.pth
QUAL_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_7class_base_vs_ha_seed42


## 2. Define experiments

Check that these checkpoint paths exist. If your output folder names differ slightly, adjust the paths here.

In [2]:
OVERVIEW_SCENARIOS = {}

for block_idx in BLOCKS_TO_COMPARE:
    block_name = "last" if block_idx == -1 else f"block{block_idx}"

    OVERVIEW_SCENARIOS[f"Base CE ({block_name})"] = {
        "checkpoint": BASE_CKPT,
        "checkpoint_model_type": "panderm",
        "csv": QUAL_CSV,
        "target_block_index": block_idx,
        "out_dir": QUAL_ROOT / f"cam_base_ce_{block_name}",
    }

    OVERVIEW_SCENARIOS[f"HA 0.75 ({block_name})"] = {
        "checkpoint": HA_CKPT,
        "checkpoint_model_type": "panderm",
        "csv": QUAL_CSV,
        "target_block_index": block_idx,
        "out_dir": QUAL_ROOT / f"cam_ha075_{block_name}",
    }


for name, cfg in OVERVIEW_SCENARIOS.items():
    print(name)
    print("  checkpoint:", cfg["checkpoint"])
    print("  csv:", cfg["csv"])
    print("  target_block_index:", cfg["target_block_index"])
    print("  out_dir:", cfg["out_dir"])

    if not cfg["checkpoint"].exists():
        print("  [WARN] missing checkpoint")
    if not cfg["csv"].exists():
        print("  [WARN] missing CSV")

Base CE (last)
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-base.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed42.csv
  target_block_index: -1
  out_dir: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_7class_base_vs_ha_seed42/cam_base_ce_last
HA 0.75 (last)
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-HA075.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed42.csv
  target_block_index: -1
  out_dir: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_7class_base_vs_ha_seed42/cam_ha075_last
Base CE (block-4)
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-base.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HA

## 3. Helper functions

In [3]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_overview_cams(
    scenarios: dict,
    img_dir: Path,
    num_samples: int,
    dry_run: bool = False,
):
    for scenario_name, cfg in scenarios.items():
        out_dir = cfg["out_dir"]
        out_dir.mkdir(parents=True, exist_ok=True)

        panel_items = (
            "rgb_gt_mask,"
            "gradcam_a,"
            "gradcam_b,"
            "map_diff,"
            "finercam"
        )

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(cfg["csv"]),
            "--image_col", "image_rel_path",
            "--img_dir", str(img_dir),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", cfg["checkpoint_model_type"],
            "--class_preset", "ham",
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--compare_mode", "gt_topk_non_target",
            "--topk_compare", "3",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--target_block_index", str(cfg["target_block_index"]),
            "--save_json",
        ]

        print(f"\nRunning overview CAM generation: {scenario_name}")
        run_command(cmd, dry_run=dry_run)

## 4. Generate CAM panels

Set `dry_run=True` first if you only want to inspect the commands.

In [4]:
generate_overview_cams(
    scenarios=OVERVIEW_SCENARIOS,
    img_dir=IMG_DIR,
    num_samples=10,
    dry_run=False,
)


Running overview CAM generation: Base CE (last)

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed42.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints3/checkpoint-best-base.pth --checkpoint_model_type panderm --class_preset ham --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_7class_base_vs_ha_seed42/cam_base_ce_last --num_samples 10 --method finercam --compare_mode gt_topk_non_target --topk_compare 3 --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -1 --save_json
[info] Loaded PanDerm Base FT from checkpoint-best-base.pth
[info] check

## 5. Write PDF config

In [7]:
CONFIG_DIR = REPO_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

overview_pdf_config = [
    {
        "name": name,
        "folder": str(cfg["out_dir"].relative_to(REPO_ROOT)),
    }
    for name, cfg in OVERVIEW_SCENARIOS.items()
]

OVERVIEW_JSON = CONFIG_DIR / f"overview_7class_base_vs_ha_blocks_seed{QUAL_SEED}.json"
OVERVIEW_JSON.write_text(json.dumps(overview_pdf_config, indent=2))

print("Saved:", OVERVIEW_JSON)
print(json.dumps(overview_pdf_config, indent=2))

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/configs/overview_7class_base_vs_ha_blocks_seed42.json
[
  {
    "name": "Base CE (last)",
    "folder": "outputs/overview_7class_base_vs_ha_seed42/cam_base_ce_last"
  },
  {
    "name": "HA 0.75 (last)",
    "folder": "outputs/overview_7class_base_vs_ha_seed42/cam_ha075_last"
  },
  {
    "name": "Base CE (block-4)",
    "folder": "outputs/overview_7class_base_vs_ha_seed42/cam_base_ce_block-4"
  },
  {
    "name": "HA 0.75 (block-4)",
    "folder": "outputs/overview_7class_base_vs_ha_seed42/cam_ha075_block-4"
  }
]


## 6. Build comparison PDF

In [8]:
def build_qualitative_pdf(
    csv_path: Path,
    experiments_json_path: Path,
    out_pdf: Path,
    num_samples: int = 10,
    dry_run: bool = False,
):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.make_qualitative_comparison_pdf",
        "--csv", str(csv_path),
        "--image_col", "image_rel_path",
        "--gt_col", "gt_label",
        "--out_pdf", str(out_pdf),
        "--experiments_json_path", str(experiments_json_path),
        "--num_samples", str(num_samples),
        "--missing_policy", "placeholder",
    ]

    run_command(cmd, dry_run=dry_run)


build_qualitative_pdf(
    csv_path=QUAL_CSV,
    experiments_json_path=OVERVIEW_JSON,
    out_pdf=QUAL_ROOT / f"overview_7class_base_vs_ha_blocks_seed{QUAL_SEED}.pdf",
    num_samples=10,
    dry_run=False,
)


python -m scripts.make_qualitative_comparison_pdf --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed42.csv --image_col image_rel_path --gt_col gt_label --out_pdf /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_7class_base_vs_ha_seed42/overview_7class_base_vs_ha_blocks_seed42.pdf --experiments_json_path /Users/choekyelnyungmartsang/Developer/master-thesis/configs/overview_7class_base_vs_ha_blocks_seed42.json --num_samples 10 --missing_policy placeholder
Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_7class_base_vs_ha_seed42/overview_7class_base_vs_ha_blocks_seed42.pdf
Pages written: 10


## Notes for interpretation

For MEL images, the green cue is present. The most important visual question is whether `Cue CE` and `Cue HA` place GradCAM / FinerCAM activation on that cue.

Expected pattern:

- `Clean CE`: no systematic focus on cue.
- `Cue CE`: likely strong focus on cue if the shortcut is learned.
- `Cue HA`: may also focus on cue because the cue is inside the lesion mask, so lesion HA does not penalize it.
